# Probability Calibration for Decision Quality

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/16_decision_thresholds_calibration_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Diagnose whether a classifier's probabilities are trustworthy using reliability diagrams and the Brier score
2. Apply post-hoc calibration with `CalibratedClassifierCV` (isotonic vs. sigmoid/Platt) and measure the improvement
3. Explain why tree-based ensembles (Random Forests, Gradient Boosting) are often miscalibrated and why linear models usually are not
4. Recognize when calibration matters for a decision — and when it does not (AUC and ranking are invariant under calibration)
5. Run a short cost-based threshold refresh from NB07, applied on top of calibrated probabilities

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: When a "70% Probability" Actually Means 70%

The **State Health Department** is ready to deploy the breast cancer screening tool — but the oncologists have a specific concern: *"When the system tells a patient their risk is 70%, that number will be printed in the chart, read aloud, and used to decide next steps. It had better actually mean 70%."*

This is a **calibration** question, not a discrimination question. AUC and ROC curves in NB07 measured how well the model *ranks* patients — whether a malignant case is scored higher than a benign one. Calibration is different: it asks whether the predicted probabilities themselves are trustworthy as probabilities. A Random Forest can have AUC = 0.99 and still be systematically overconfident — predicting 0.85 when the true rate is 0.65. For ranking decisions (screening priority lists, Kaggle submissions scored by AUC), calibration does not matter. For *action* decisions (how to counsel a patient, how to price risk, how much to spend on intervention), it matters enormously.

Tree-based ensembles are notorious for miscalibration: averaging binary tree votes compresses probabilities away from the extremes. Linear models (LogReg with properly tuned regularization) are usually close to calibrated out of the box. This notebook shows how to diagnose the gap and fix it.

> **Today's focus:** reliability diagrams, Brier score, `CalibratedClassifierCV` with isotonic and sigmoid methods, and when calibration is the difference between a model that informs decisions and one that misleads them. NB07 covered threshold tuning under a cost matrix; this notebook adds a short refresh and then spends its time where NB07 could not — on the probabilities themselves.

> **A question that often comes up here:** *"ROC-AUC is already 0.99 — why do we care about calibration at all?"* Because ROC-AUC and calibration answer different questions. ROC-AUC asks *"does the model rank a random malignant case higher than a random benign case?"* (a *ranking* property). Calibration asks *"when the model says 70%, is the true rate really 70%?"* (a *probability* property). A model can have a perfect ROC-AUC and miscalibrated probabilities — the ranking is right but the probability scale is off. For pure ranking decisions (priority lists, Kaggle AUC leaderboards), calibration does not matter. For *action* decisions tied to probability thresholds or expected cost calculations, it matters enormously.

---


## 1. Setup

Before the screening tool can go live, the Health Department needs to answer one operational question: at what probability threshold should the system recommend a patient for biopsy? This cell imports the toolkit for answering that question — scikit-learn's calibration utilities (`calibration_curve`, `CalibratedClassifierCV`), standard classification metrics, and plotting libraries. We lock `RANDOM_SEED = 474` so that every threshold sweep and calibration curve is fully reproducible when regulators or clinicians ask to verify the analysis.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

print("✓ Setup complete!")

**Reading the output:**

The `Setup complete!` confirmation with **RANDOM_SEED = 474** means every import resolved and the global seed is locked. Notice the two calibration-specific imports: `calibration_curve` (for diagnosing whether the model's probabilities match reality) and `CalibratedClassifierCV` (for correcting them if they do not). These are the two halves of the calibration workflow — diagnose first, then fix.

**Why this matters:** When clinicians see a predicted probability of 0.70, they interpret it as "70% chance of malignancy." If the model is miscalibrated — predicting 0.70 when the true rate is only 0.50 — clinicians will over-refer patients to biopsy, wasting resources and causing unnecessary anxiety. Clean setup is the first reproducibility checkpoint for an analysis that will directly influence clinical decisions.

> **A question that often comes up here:** *"Why does `calibration_curve` need probabilities instead of hard predictions?"* Because calibration *is* a question about probabilities — specifically, about whether the predicted probability at a given confidence level matches the observed positive rate at that level. Hard 0/1 predictions contain no information about confidence, so you cannot ask calibration questions about them. Whenever you want to check calibration, you need `predict_proba` or `decision_function`, not `predict`.

---


## 2. Load Data and Train Model

To illustrate threshold selection and calibration in a controlled setting, we generate a synthetic binary-classification dataset (5,000 samples, 20 features, 70/30 class imbalance, 5% label noise). The 70/30 imbalance mirrors the screening scenario where most patients are healthy (negative) and a smaller fraction have malignancies (positive). The 5% noise simulates the diagnostic uncertainty clinicians face — some biopsies come back indeterminate, and some benign lesions initially look suspicious.

After splitting 60/20/20, a Random Forest is trained on the training set and its predicted probabilities on the validation set become the raw material for every threshold and calibration analysis that follows.

> 💡 **Gemini Prompt:** "Generate synthetic binary classification: 5000 samples, 20 features (15 informative, 5 redundant), weights=[0.7,0.3], 5% noise, seed 474. Split 60/20/20. Print sizes and class distribution."
>
> **After running, verify:**
> - Train 3000, Val 1000, Test 1000
> - Class distribution \~70/30
> - 20 features total
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Generate classification dataset
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=15,
    n_redundant=5, weights=[0.7, 0.3], flip_y=0.05,
    random_state=RANDOM_SEED
)

# Split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Class distribution (validation): {np.bincount(y_val)}")

**Reading the output:**

The printout confirms **Train / Val / Test** sizes of roughly **3,000 / 1,000 / 1,000**, matching the 60/20/20 split. The class distribution in the validation set should reflect the \~70/30 imbalance: approximately 700 negatives and 300 positives. In the screening context, this means most patients are healthy, but the 30% positive rate is high enough that a threshold sweep can explore a meaningful range of sensitivity-specificity trade-offs.

**Key takeaway:** Always print split sizes and class counts immediately after splitting. If the minority class were extremely rare (<5%), even a model predicting "all negative" would look accurate — the imbalance here is moderate enough that the default 0.50 threshold is a reasonable starting point to improve upon.

> **A question that often comes up here:** *"Why generate synthetic data here instead of using Breast Cancer like the previous notebooks?"* Because the calibration story needs a model that is visibly miscalibrated, and a Random Forest on 5,000 synthetic points delivers that reliably. Breast Cancer with Logistic Regression tends to be well-calibrated already (logistic regression optimizes log-loss, which pushes calibration as a side effect). The synthetic setup gives you a clear before-and-after picture, which is what the pedagogy needs.

---


> 💡 **Gemini Prompt:** "Train RandomForestClassifier (100 trees, max_depth=10, seed 474). Get predicted probabilities on validation. Print ROC-AUC."
>
> **After running, verify:**
> - Model fitted on training data
> - y_val_proba has positive class probabilities
> - ROC-AUC printed (above 0.5)
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Train Random Forest classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Get predicted probabilities
y_val_proba = rf_model.predict_proba(X_val)[:, 1]

print(f"\nROC-AUC Score: {roc_auc_score(y_val, y_val_proba):.4f}")

**Reading the output:**

The **ROC-AUC** score on the validation set tells you how well the Random Forest separates positive cases (malignancies) from negatives (benign) *across all possible thresholds*. A value close to 1.0 means the model's probability distributions for the two classes are well separated. However, high AUC does **not** guarantee that the default 0.50 threshold is the right clinical decision — that depends on the relative cost of missing a cancer (false negative) versus alarming a healthy patient (false positive).

**Why this matters:** AUC summarises discrimination ability, but the operating point the Health Department adopts must be driven by the cost matrix, not by AUC alone. A model with AUC = 0.95 could still be deployed at a terrible threshold if the cost structure is ignored. The next section translates AUC into dollars by defining exactly what each type of error costs the screening programme.

> **A question that often comes up here:** *"ROC-AUC is high — doesn't that already tell me the model is good?"* It tells you the ranking is good. It tells you *nothing* about whether the probabilities are trustworthy as probabilities. This is the mental-model shift NB16 is built around: high AUC + miscalibrated probabilities is a common combination, especially for tree-based ensembles. The calibration diagnostic (Section 4) measures what AUC cannot.

---


## 3. Recap from NB07 — Thresholds and Costs (5-minute refresh)

> 📎 **Refresh, not a full tour.** NB07 already built the threshold-tuning workflow from scratch: confusion matrix, cost matrix, threshold sweep, optimal-threshold selection, and sensitivity analysis. This notebook applies that same workflow in Sections 3.1–3.2 as a quick refresher, *then* pivots to its real subject — **calibration** (Section 4). If you already remember NB07 fluently, skim this section and spend your time on Section 4.

### 3.1 Define Cost Matrix

The Health Department's medical director and finance officer agree on a cost structure for the screening programme. Each prediction outcome carries a different consequence:

- **True Positive (TP):** Correctly flagging a malignancy for biopsy → early treatment saves a life → value: **+\$100**
- **False Positive (FP):** Flagging a healthy patient → unnecessary biopsy, anxiety, wasted lab time → cost: **-\$30**
- **False Negative (FN):** Missing a malignancy → delayed diagnosis, worse outcomes → cost: **-\$150**
- **True Negative (TN):** Correctly clearing a healthy patient → no action needed → value: **\$0**

The critical asymmetry: missing a cancer (FN = -\$150) costs **five times** more than a false alarm (FP = -\$30). This asymmetry will push the optimal threshold *below* 0.50 — the screening programme should accept more false alarms to avoid the catastrophic cost of missed cancers.

> 💡 **Gemini Prompt:** "Define cost matrix: TP=+\$100, FP=-\$30, FN=-\$150, TN=\$0. Write compute_expected_value(y_true, y_pred, costs) that returns total value and confusion counts dict."
>
> **After running, verify:**
> - Cost matrix printed: TP=\$100, FP=-\$30, FN=-\$150, TN=\$0
> - Function returns total_value and counts dict
> - FN is most expensive at -\$150
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Cost matrix
COST_MATRIX = {
    'TP': 100,   # Benefit
    'FP': -30,   # Cost
    'FN': -150,  # Cost
    'TN': 0      # No action
}

def compute_expected_value(y_true, y_pred, cost_matrix):
    """Compute expected value given cost matrix"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    total_value = (
        tp * cost_matrix['TP'] +
        fp * cost_matrix['FP'] +
        fn * cost_matrix['FN'] +
        tn * cost_matrix['TN']
    )
    
    return total_value, {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}

print("Cost Matrix Defined:")
for key, value in COST_MATRIX.items():
    print(f"  {key}: ${value}")

**Reading the output:**

The printed cost matrix confirms **TP = +$100**, **FP = -$30**, **FN = -$150**, and **TN = $0**. Two things jump out: missing a cancer (FN) costs five times more than a false alarm (FP), and correctly clearing a healthy patient adds no monetary value. This asymmetry is the single most important input to threshold selection — it means the optimal threshold will be *below* 0.50, because the screening programme should err on the side of caution.

The `compute_expected_value` function encodes this cost structure into a reusable calculator: given true labels and predictions, it multiplies each confusion-matrix cell by its cost and returns the total dollar value. This function will be called at every threshold in the sweep that follows.

**Key takeaway:** Before touching any threshold, always map the cost matrix with the domain stakeholder (here, the medical director). The ratio of FN cost to FP cost — 150/30 = 5:1 — is the single most important driver of where the optimal threshold lands.

> **A question that often comes up here:** *"Where does the 5:1 ratio come from — is it data-driven?"* No, the cost matrix comes from the business, not the data. The stakeholder tells you what a false negative costs (missed cancer → delayed treatment → \$150 equivalent in the simulation) and what a false positive costs (unnecessary biopsy → \$30). Your job is not to estimate those numbers — your job is to use them correctly once they are given. The cost matrix is the place where business judgement enters the modeling pipeline; treat the numbers as inputs, not parameters to tune.

---


### 3.2 Threshold Sweep with Expected Cost

Instead of accepting the default 0.50 decision boundary — which treats false positives and false negatives as equally costly — we sweep thresholds from 0.10 to 0.85 and compute the total expected value under the Health Department's cost matrix at each point. The threshold that maximises total value is the one the screening programme should adopt. This is where the model stops being a statistical exercise and becomes a clinical policy decision.

> 💡 **Gemini Prompt:** "Sweep thresholds 0.10 to 0.85 (step 0.05). At each, binarize probabilities, compute expected value via cost matrix. Find optimal threshold. Print top 10 results."
>
> **After running, verify:**
> - Table shows threshold, total_value, avg_value, TP, FP, FN, TN
> - Optimal threshold printed at bottom
> - Lower thresholds catch more positives
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Sweep thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
results = []

for threshold in thresholds:
    y_pred = (y_val_proba >= threshold).astype(int)
    total_value, counts = compute_expected_value(y_val, y_pred, COST_MATRIX)
    
    results.append({
        'threshold': threshold,
        'total_value': total_value,
        'avg_value_per_case': total_value / len(y_val),
        'TP': counts['TP'],
        'FP': counts['FP'],
        'FN': counts['FN'],
        'TN': counts['TN']
    })

results_df = pd.DataFrame(results)
best_threshold = results_df.loc[results_df['total_value'].idxmax(), 'threshold']

print("\n=== THRESHOLD SWEEP RESULTS ===")
print(results_df.head(10))
print(f"\nOptimal Threshold (by total value): {best_threshold:.2f}")

**Reading the output:**

The results table lists each threshold alongside the **total expected value** and the four confusion-matrix counts. The optimal threshold (printed at the bottom) maximises total value and is typically *below* 0.50 — the 5:1 FN-to-FP cost ratio pushes the model to cast a wider net, catching nearly all true malignancies at the expense of more false alarms.

The `avg_value_per_case` column translates this into a per-patient figure the Health Department can quote: "Under the recommended threshold, each screening decision generates an expected net value of $X." As the threshold rises, FN count increases (missed cancers) and the expected value drops — the programme is saving on biopsy costs but losing far more on missed diagnoses.

**Why this matters:** Picking a threshold by accuracy alone would select the point that minimises total errors, regardless of their severity. Cost-based optimisation picks the point that minimises total *harm* — a fundamentally different question that the screening programme's stakeholders actually care about.

> **A question that often comes up here:** *"The optimal threshold came in at 0.35 — why is it not 0.5?"* Because 0.5 is the right threshold only when false positive and false negative costs are equal. They almost never are. A 5:1 FN:FP cost ratio pushes the optimum *below* 0.5, because you want to flag more patients (accepting more false positives) to catch more cancers. Every threshold other than 0.5 is an operational choice encoded by the cost asymmetry — and documenting *why* you chose 0.35 is more important than the number itself.

---


> 💡 **Gemini Prompt:** "Create 1x2 subplot: left plots total expected value vs threshold with red line at optimal; right plots TP, FP, FN counts vs threshold."
>
> **After running, verify:**
> - Left panel peaks at optimal threshold
> - Right panel has three lines (TP, FP, FN)
> - Both share same x-axis range
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Visualize threshold impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total value vs threshold
axes[0].plot(results_df['threshold'], results_df['total_value'], marker='o')
axes[0].axvline(best_threshold, color='r', linestyle='--', label=f'Optimal: {best_threshold:.2f}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Total Expected Value ($)')
axes[0].set_title('Expected Value vs Threshold')
axes[0].legend()
axes[0].grid(True)

# Confusion matrix components
axes[1].plot(results_df['threshold'], results_df['TP'], marker='o', label='True Positives')
axes[1].plot(results_df['threshold'], results_df['FP'], marker='s', label='False Positives')
axes[1].plot(results_df['threshold'], results_df['FN'], marker='^', label='False Negatives')
axes[1].axvline(best_threshold, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Count')
axes[1].set_title('Error Counts vs Threshold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

**Reading the output:**

The **left panel** shows the expected-value curve: it rises, peaks at the optimal threshold (red dashed line), and then falls as the threshold becomes too conservative and starts missing malignancies. The peak is the screening programme's "sweet spot" — the point where every additional false alarm costs exactly as much as the avoided missed cancers save.

The **right panel** decomposes the confusion-matrix counts: as the threshold increases, true positives and false positives both drop (fewer referrals overall), while false negatives climb (more missed cancers). The optimal threshold sits where the monetary gain from reducing false alarms is exactly offset by the mounting cost of new missed diagnoses.

**Key takeaway:** Presenting value *and* error counts side by side lets the Health Department's medical director understand the recommendation intuitively: "We chose this threshold because lowering it further would add false alarms faster than it saves missed cancers, and raising it would miss cancers faster than it saves on biopsies."

> **A question that often comes up here:** *"The curve peaks and then falls — what does the falling side represent?"* The falling side is the regime where the model has become too conservative. Above the optimal threshold, you stop flagging marginal cases, which means you save on false-positive biopsies but start missing cancers — and missing cancers is five times more expensive than false alarms. The peak is the sweet spot where the two effects balance. Stakeholders often intuit that a higher threshold is "more cautious," but on an asymmetric cost structure, *too cautious* is dangerous too.

---


## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Select a threshold that minimizes expected cost and justify it.

**Instructions:**
1. Review the threshold sweep results above
2. Identify the threshold that maximizes expected value
3. Explain why this threshold makes business sense
4. Discuss what tradeoffs are being made

---

### YOUR THRESHOLD RECOMMENDATION HERE:

**Recommended Threshold:**  
[Value and reasoning]

**Business Justification:**  
[Why this threshold makes sense for the business]

**Tradeoffs:**  
[What are we gaining vs losing at this threshold?]

---

## 4. Calibration: Are Probabilities Trustworthy?

### 4.1 Calibration Plot

The threshold analysis above assumed the model's predicted probabilities are meaningful — that a predicted 0.70 actually corresponds to a 70% chance of malignancy. But Random Forests are notorious for producing probabilities that cluster near 0.0 and 1.0, compressing the mid-range where clinical decisions are most uncertain. If the oncologist sees "70% probability" but the true rate is only 50%, she will refer too many patients to biopsy, overwhelming the lab and causing unnecessary patient anxiety.

A reliability diagram tests this directly by binning predictions and comparing predicted probabilities to observed positive rates.

> 💡 **Gemini Prompt:** "Compute calibration curve (10 bins). Create 1x2 subplot: reliability diagram comparing RF to perfect diagonal (left); histogram of predicted probabilities by actual class (right)."
>
> **After running, verify:**
> - Reliability diagram shows points near/deviating from diagonal
> - Histogram shows two overlapping distributions for class 0 and 1
> - Interpretation notes about under/overconfidence
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compute calibration curve
prob_true, prob_pred = calibration_curve(y_val, y_val_proba, n_bins=10, strategy='uniform')

# Plot calibration
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration plot
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[0].plot(prob_pred, prob_true, marker='o', label='Random Forest')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction of Positives')
axes[0].set_title('Calibration Plot')
axes[0].legend()
axes[0].grid(True)

# Probability histogram
axes[1].hist(y_val_proba[y_val == 0], bins=30, alpha=0.5, label='Negative Class', edgecolor='black')
axes[1].hist(y_val_proba[y_val == 1], bins=30, alpha=0.5, label='Positive Class', edgecolor='black')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Predicted Probability Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n⚠️ Calibration Assessment:")
print("  - Points close to diagonal = well-calibrated")
print("  - Points below diagonal = overconfident")
print("  - Points above diagonal = underconfident")

**Reading the output:**

The **left panel** (reliability diagram) plots the actual fraction of positives against the model's predicted probability in each bin. Points on the diagonal mean perfect calibration. Points *below* the diagonal mean the model is **overconfident** — it says 0.80 but the true rate is only 0.65. Points *above* mean **underconfidence**. Random Forests typically show an S-shaped deviation: overconfident at high probabilities and underconfident at low probabilities, because averaging binary tree votes compresses the probability range.

The **right panel** shows how predicted probabilities distribute across the two classes. Well-separated histograms confirm good discrimination (the model can tell the classes apart), but the shape of the distribution reveals calibration issues — if most predictions cluster near 0.0 and 1.0 with a gap in the middle, the model is confident but not always correctly confident.

**Why this matters:** For the Health Department, clinicians will treat the predicted probability as a genuine risk estimate. If the model says "70% chance of malignancy" but the actual rate is 50%, the screening programme will over-refer patients — each unnecessary biopsy costs the system money and the patient stress. Calibration must be verified before deployment.

> **A question that often comes up here:** *"What exactly does the diagonal in a reliability diagram represent?"* It represents perfect calibration: at every predicted probability value on the x-axis, the corresponding observed positive rate on the y-axis is exactly that same value. A model that predicts 0.7 for 100 patients, of whom exactly 70 are truly positive, falls on the diagonal. A model whose 0.7 predictions correspond to only 50% actual positives falls below the diagonal — it is overconfident. Tree ensembles typically show an S-shape: overconfident at high predictions and underconfident at low predictions, because averaging binary tree votes compresses the probability range.

---


### 4.2 Apply Calibration

When the reliability diagram reveals systematic miscalibration, we wrap the trained classifier in `CalibratedClassifierCV` with isotonic regression. This post-hoc adjustment maps the Random Forest's raw scores to better-calibrated probabilities without retraining the base model — think of it as a translation layer that converts "RF confidence" into "actual probability." We fit the calibrator on the validation set and evaluate on the held-out test set to avoid inflating calibration quality by testing on the same data used for fitting.

> 💡 **Gemini Prompt:** "Apply isotonic calibration with CalibratedClassifierCV(method='isotonic', cv='prefit'). Plot original and calibrated curves together on reliability diagram."
>
> **After running, verify:**
> - Overlay shows diagonal, original RF, calibrated RF
> - Calibrated curve closer to diagonal
> - Confirmation of isotonic calibration
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier

# Calibration via CalibratedClassifierCV with cv=5 — the calibrator is fit using
# internal 5-fold CV on the training set (base estimator refit per fold). No
# X_val or X_test is used for FITTING the calibrator; we evaluate Brier score
# and the reliability diagram on X_val. The test set stays locked for NB14.
base_estimator = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1
)

calibrated_iso = CalibratedClassifierCV(base_estimator, method='isotonic', cv=5)
calibrated_iso.fit(X_train, y_train)
y_cal_iso_proba = calibrated_iso.predict_proba(X_val)[:, 1]

calibrated_sig = CalibratedClassifierCV(base_estimator, method='sigmoid', cv=5)
calibrated_sig.fit(X_train, y_train)
y_cal_sig_proba = calibrated_sig.predict_proba(X_val)[:, 1]

# Original (uncalibrated) probabilities on X_val for a fair three-way Brier comparison
y_val_proba_orig = rf_model.predict_proba(X_val)[:, 1]

# --- Brier score on the validation set (lower is better) ---
brier_original = brier_score_loss(y_val, y_val_proba_orig)
brier_isotonic = brier_score_loss(y_val, y_cal_iso_proba)
brier_sigmoid  = brier_score_loss(y_val, y_cal_sig_proba)

print("=== BRIER SCORE COMPARISON on X_val (lower = better) ===")
print(f"Original Random Forest:     {brier_original:.4f}")
print(f"Isotonic-calibrated RF:     {brier_isotonic:.4f}")
print(f"Sigmoid-calibrated RF:      {brier_sigmoid:.4f}")

if brier_isotonic <= brier_sigmoid:
    y_cal_proba = y_cal_iso_proba
    winner = 'Isotonic'
else:
    y_cal_proba = y_cal_sig_proba
    winner = 'Sigmoid'
print(f"\n→ Winner by Brier: {winner} calibration")

# --- Reliability diagram on X_val comparing original vs both calibrators ---
prob_true_iso, prob_pred_iso = calibration_curve(y_val, y_cal_iso_proba, n_bins=10, strategy='uniform')
prob_true_sig, prob_pred_sig = calibration_curve(y_val, y_cal_sig_proba, n_bins=10, strategy='uniform')
prob_true_orig, prob_pred_orig = calibration_curve(y_val, y_val_proba_orig, n_bins=10, strategy='uniform')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
ax.plot(prob_pred_orig, prob_true_orig, marker='o',
        label=f'Original RF (Brier {brier_original:.3f})', alpha=0.7)
ax.plot(prob_pred_iso, prob_true_iso, marker='s',
        label=f'Isotonic (Brier {brier_isotonic:.3f})', alpha=0.7)
ax.plot(prob_pred_sig, prob_true_sig, marker='^',
        label=f'Sigmoid (Brier {brier_sigmoid:.3f})', alpha=0.7)
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration on Validation Set — Original vs Isotonic vs Sigmoid')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f"\n✓ Calibration fit via CalibratedClassifierCV(cv=5) on X_train; Brier + reliability evaluated on X_val.")
print(f"  Pick the calibrator with the lower Brier score — here: {winner}.")
print("  The test set stays locked for NB14's ceremony.")


**Reading the output:**

The Brier score is the mean squared error between predicted probabilities and true labels, averaged over all test samples. Lower is better; 0 is a perfect probabilistic classifier. Tree-based ensembles like Random Forest typically have higher Brier scores than well-tuned linear models for exactly the miscalibration reason shown in the reliability diagram above — their predicted probabilities cluster near 0 and 1 even when the true rate is in the middle.

The three-way comparison lets you pick the right calibrator:

- **Isotonic regression** is non-parametric and flexible. It fits the reliability curve's shape but needs enough validation data to estimate it stably. With small validation sets (< 1,000 samples) it can overfit the calibration curve. On this 1,000-sample validation set it usually wins.
- **Sigmoid / Platt scaling** fits a 2-parameter logistic correction. It is less flexible but far more stable on small data. Use it when the validation set is modest and the reliability curve is roughly monotonic.
- **Neither** — if the Brier score barely moves after calibration, the original model was already well-calibrated (typical for LogReg with reasonable regularization; atypical for RF / GBM).

Critically, calibration does **not** change the model's ranking of patients (AUC is unchanged). A patient ranked higher-risk than another before calibration is still ranked higher after. What changes is the *scale* of the probabilities — so "70% probability" starts meaning something close to 70%.

**Key takeaway:** Report both AUC (discrimination) and Brier (calibration) to stakeholders. A model with AUC 0.99 and Brier 0.20 is an excellent ranker and a terrible probability estimator. A model with AUC 0.97 and Brier 0.05 is a slightly worse ranker but a *usable* probability estimator. The Health Department needs the second one.

> **A question that often comes up here:** *"Do I always need to pick between isotonic and sigmoid calibration?"* In practice, run both and pick the lower Brier score on held-out data — which is exactly what the code does. Rule of thumb: isotonic is more flexible but needs more validation data (1,000+ rows before it shines); sigmoid is more stable on smaller validation sets (under 1,000 rows) and produces a smoother correction curve. If neither moves the Brier score noticeably, your model was already well-calibrated — which happens to logistic regression with good regularization, and is why NB16's dramatic demos are on tree ensembles, not logistic models.

---


## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Check calibration and decide whether calibration is needed.

**Instructions:**
1. Review the calibration plots above
2. Assess whether the model is well-calibrated
3. Decide if calibration would improve decision-making
4. Justify your recommendation

---

### YOUR CALIBRATION ASSESSMENT HERE:

**Calibration Quality:**  
[Is the model well-calibrated? What patterns do you see?]

**Recommendation:**  
[Should we use calibrated probabilities?]

**Justification:**  
[Why or why not? What's the impact on decision-making?]

---

## 5. Decision Policy Summary

### 5.1 Final Recommendation

The screening programme needs a single, defensible policy document that the medical director can sign off on: the chosen threshold, the expected outcomes at that threshold, and the evidence supporting the choice. This cell combines the optimal threshold, total expected value, confusion-matrix counts, and classification report into one summary that can be presented to the Health Department's board.

> 💡 **Gemini Prompt:** "Apply optimal threshold to validation probabilities. Print decision policy with threshold, expected value, confusion matrix, and classification_report."
>
> **After running, verify:**
> - Optimal threshold and total value in dollars
> - All four CM counts listed
> - Full classification_report displayed
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Apply optimal threshold
y_val_pred_optimal = (y_val_proba >= best_threshold).astype(int)
total_value, counts = compute_expected_value(y_val, y_val_pred_optimal, COST_MATRIX)

print("=== DECISION POLICY RECOMMENDATION ===")
print(f"\nOptimal Threshold: {best_threshold:.2f}")
print(f"Expected Total Value: ${total_value:,.2f}")
print(f"Expected Value per Case: ${total_value/len(y_val):,.2f}")
print(f"\nConfusion Matrix at Optimal Threshold:")
print(f"  True Positives: {counts['TP']}")
print(f"  False Positives: {counts['FP']}")
print(f"  False Negatives: {counts['FN']}")
print(f"  True Negatives: {counts['TN']}")

print(f"\nClassification Report:")
print(classification_report(y_val, y_val_pred_optimal))

**Reading the output:**

The summary prints the **optimal threshold**, the **total expected value** in dollars, and the per-case average — all three views the Health Department needs. The confusion-matrix breakdown shows exactly how many patients are correctly flagged (TP), falsely alarmed (FP), missed (FN), and correctly cleared (TN). The classification report adds precision, recall, and F1 for both classes.

Quoting all three views — dollars for the finance officer, rates for the oncologist, counts for the operations manager — makes the recommendation accessible to every stakeholder in the room. The medical director can then sign the policy document knowing that the threshold was chosen to minimise total patient harm, not just to maximise a statistical score.

**Why this matters:** A decision policy must be communicated in units each stakeholder cares about. "We recommend threshold 0.35" means nothing to a clinician; "At this threshold, we catch 93% of cancers while sending 15% of healthy patients for unnecessary biopsy" tells the whole story.

> **A question that often comes up here:** *"How do I turn this expected-value calculation into something the medical director can sign off on?"* Translate dollar-optimal into rate-optimal: "at the chosen threshold, we catch X% of cancers and flag Y% of healthy patients for unnecessary follow-up; the expected value per screening is \$Z." Three audiences, three framings, same number. The director wants the recall rate; the finance officer wants the dollars per screening; the operations team wants the false-positive rate translating into biopsy workload. A good decision memo quotes all three.

---


### 5.2 Sensitivity Analysis

The cost matrix reflects the Health Department's current best estimates, but those estimates carry uncertainty. What if the true cost of a missed cancer is $200 instead of $150? What if improved biopsy procedures reduce the FP cost to $20? A sensitivity analysis varies the false-negative cost over a plausible range ($100–$250) and records how the optimal threshold and expected value shift. If the threshold barely moves, the policy is robust; if it swings dramatically, the board needs tighter cost estimates before committing.

> 💡 **Gemini Prompt:** "Sensitivity analysis: vary FN cost [100,150,200,250]. For each, re-sweep thresholds to find optimal. Plot optimal threshold and expected value vs FN cost."
>
> **After running, verify:**
> - Table has 4 rows with FN_Cost, Optimal_Threshold, Expected_Value
> - Higher FN cost pushes threshold lower
> - Two plots show the tradeoff
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Test sensitivity to cost assumptions
fn_costs = [100, 150, 200, 250]
sensitivity_results = []

for fn_cost in fn_costs:
    temp_cost_matrix = COST_MATRIX.copy()
    temp_cost_matrix['FN'] = -fn_cost
    
    # Find optimal threshold for this cost
    best_value = -np.inf
    best_thresh = 0.5
    
    for threshold in thresholds:
        y_pred = (y_val_proba >= threshold).astype(int)
        value, _ = compute_expected_value(y_val, y_pred, temp_cost_matrix)
        if value > best_value:
            best_value = value
            best_thresh = threshold
    
    sensitivity_results.append({
        'FN_Cost': fn_cost,
        'Optimal_Threshold': best_thresh,
        'Expected_Value': best_value
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

print("\n=== SENSITIVITY ANALYSIS ===")
print("How does optimal threshold change with FN cost?")
print(sensitivity_df)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sensitivity_df['FN_Cost'], sensitivity_df['Optimal_Threshold'], marker='o')
axes[0].set_xlabel('False Negative Cost ($)')
axes[0].set_ylabel('Optimal Threshold')
axes[0].set_title('Threshold Sensitivity to FN Cost')
axes[0].grid(True)

axes[1].plot(sensitivity_df['FN_Cost'], sensitivity_df['Expected_Value'], marker='o')
axes[1].set_xlabel('False Negative Cost ($)')
axes[1].set_ylabel('Expected Value ($)')
axes[1].set_title('Expected Value vs FN Cost')
axes[1].grid(True)

plt.tight_layout()
plt.show()

**Reading the output:**

The sensitivity table shows how the optimal threshold and expected value change as the **false-negative cost** varies from $100 to $250. As FN cost increases, the optimal threshold drops — the screening programme becomes more aggressive about flagging patients because the penalty for missing a cancer grows. If the threshold barely shifts across this range (e.g., stays between 0.30 and 0.35), the policy is robust to cost-estimation uncertainty and the board can approve it with confidence.

The two plots reinforce this: a flat threshold line means stability; a steep slope means the board must invest in more precise cost estimates before locking in a threshold. In cancer screening, where the human cost of a missed diagnosis is genuinely uncertain, presenting this sensitivity analysis alongside the point recommendation is essential for responsible deployment.

**Key takeaway:** Never present a single optimal threshold without showing how sensitive it is to the assumptions. Sensitivity analysis transforms a model recommendation into a trustworthy clinical policy that the Health Department can defend to regulators, patients, and the public.

> **A question that often comes up here:** *"How sensitive is 'sensitive'? When should the sensitivity analysis make me pause?"* Rule of thumb: if the optimal threshold moves by more than roughly 0.05 across the plausible range of the cost parameter, the policy is *cost-sensitive* and the stakeholder needs to sign off on a more confident estimate of the cost before you deploy. If the threshold moves less than 0.02 across the full plausible range, the policy is robust and you can deploy with the current best estimate. In between, report the range as a confidence interval on the threshold itself — same spirit as the CV 95% CI from NB08.

---


## 6. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Discrimination vs. calibration are two separate questions.** A model can have a perfect ROC-AUC and still produce probabilities that lie about their confidence. ROC-AUC measures ranking; calibration measures whether "70%" means 70%. Report both.
2. **Reliability diagrams are the diagnostic.** Plot the predicted probability against the observed positive rate in binned slices. Points on the diagonal are perfectly calibrated; off-diagonal points reveal overconfidence (below) or underconfidence (above) at specific probability ranges.
3. **Brier score is the one-number summary.** Lower is better; 0 is perfect. Use it to rank calibration methods (original vs. isotonic vs. sigmoid) and to report a single number to stakeholders who want to track calibration quality over time.
4. **Isotonic for more validation data, sigmoid for less.** Isotonic is a non-parametric monotone fit that captures complex shapes but overfits small samples. Sigmoid (Platt scaling) is a two-parameter logistic correction — less flexible, more stable. Run both when possible; pick by Brier.
5. **Calibration does not change AUC.** The ranking of predictions is invariant under monotone transformation, which is what calibration applies. Which means: calibration is a pure probability correction, not a discrimination change. You do not lose ranking power by calibrating.
6. **Cost-based threshold tuning + calibration together translate probabilities into decisions.** NB07 introduced the cost matrix; NB16 connects it to a trustworthy probability scale. The output of the pipeline now is: *calibrated probability → expected-cost sweep → chosen threshold → decision rule → monitored over time*.

### Critical Rules:

> **"Check calibration separately from AUC — they are different questions."**

> **"Report Brier score alongside AUC whenever probabilities inform decisions."**

> **"Run sensitivity analysis on cost parameters — the threshold is a function of the business's estimate, not a fact."**

### Next Steps:

- **NB17 (Fairness + Model Cards)** asks the question that calibrated probabilities cannot answer on their own: are the model's errors distributed fairly across groups? Slice-based evaluation, demographic parity, equal opportunity — and the model card that documents every limitation this notebook identified.
- **NB18 (Reproducibility + Monitoring + Kaggle)** asks how to actually deploy and monitor what you built. Calibration can drift over time; the monitoring plan needs a Brier-score drift signal alongside the performance signals.

> **A question that often comes up here:** *"If I only care about ROC-AUC for my final Kaggle submission, do I need to calibrate?"* Not for the Kaggle leaderboard — ROC-AUC is invariant under calibration, so you cannot improve the leaderboard score by calibrating. But for your project write-up and for every real business deployment, calibration is non-negotiable the moment a threshold or expected-cost calculation enters the picture. Calibration is the "professional polish" that separates a Kaggle submission from a production model.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- Provost, F., & Fawcett, T. (2013). *Data Science for Business*. O'Reilly Media.
- scikit-learn User Guide: [Probability Calibration](https://scikit-learn.org/stable/modules/calibration.html)
- Niculescu-Mizil, A., & Caruana, R. (2005). "Predicting good probabilities with supervised learning." *ICML*.
- Zadrozny, B., & Elkan, C. (2001). "Obtaining calibrated probability estimates from decision trees and naive Bayesian classifiers." *ICML*.

---




<center>

Thank you!

</center>